# OncoSeg — 3D Tumor Segmentation → RECIST Response, verified on Colab

This notebook **leads with a visual results gallery** (§3: real segmentation on brain MRI, accuracy, uncertainty),
then runs the full **post-fix verification suite** (§4–§8) on a Colab **GPU**, and ends with a live RECIST
response demo. No dataset or checkpoint needed for §1–§8.

**Before running:** `Runtime → Change runtime type → Hardware accelerator → GPU`, then **Runtime → Run all**.

> **One-time kernel restart (expected).** §1 installs `monai`/`numpy` etc., but Colab keeps the *old* numpy loaded
> in the running kernel, which would break later imports (`cannot import name _center`). So §1 **restarts the
> kernel once** right after installing — when it does, just **Run all again**. It's idempotent (the clone is
> skipped, pip is a no-op, and it won't restart a second time). §3 (gallery), §7, §8 run in-kernel so their figures
> display inline; §4/§6/§7-stats run as fresh subprocesses.

## 1 · Clone + install + (one-time) kernel restart

Installs `.[dev,serve,dicom]` — `monai[all]`, `nibabel`, `fastapi`, `python-multipart`, `pydicom`/`highdicom`,
`pytest`, `ruff` (the same extras CI uses, plus `dicom`). Takes ~2–3 min the first time.


In [ ]:
import os
REPO = "https://github.com/danielchen26/OncoSeg-3D-Multi-Scale-Tumor-Segmentation-for-Automated-Treatment-Response-Assessment.git"
BRANCH = "fix/review-findings"
FLAG = "/content/.oncoseg_installed"   # a FILE flag survives a kernel restart (env vars do NOT)
if not os.path.isdir("/content/oncoseg"):
    !git clone --branch $BRANCH --depth 1 $REPO /content/oncoseg
%cd /content/oncoseg
!git log --oneline -1
if not os.path.exists(FLAG):
    !pip -q install -e "/content/oncoseg[dev,serve,dicom]"
    open(FLAG, "w").close()
    print("\n>>> Installed. Restarting the kernel ONCE so fresh numpy/monai load — then run Runtime ▸ Run all again. <<<")
    import time; time.sleep(1)
    os.kill(os.getpid(), 9)   # hard-restart the Colab kernel
else:
    print("Dependencies already installed and kernel restarted — continuing.")

## 2 · Runtime + import check (after restart)


In [ ]:
import os; os.chdir('/content/oncoseg') if os.path.isdir('/content/oncoseg') else None
import sys, platform
print('Python:', sys.version.split()[0], '|', platform.platform())
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('--- optional deps ---')
for m in ['monai','nibabel','fastapi','pydicom','highdicom','scipy','numpy']:
    try: __import__(m); print(f'  {m}: OK')
    except Exception as e: print(f'  {m}: MISSING ({e})')

## 3 · Results Gallery — what the model produces (committed figures)

**Lead with the results.** Everything below is a **static PNG committed to the repo** from a single 50-epoch training run (OncoSeg 3.7M params, `embed_dim=24`). The repo ships **no checkpoint** (finding F10), so these figures are **displayed, not regenerated** — they let you inspect real validation output right now, before the verification suite (§4–§8) proves the *code paths* are correct on live tensors.

Three things to look at:

1. **3.1 Segmentation** — OncoSeg vs the UNet3D baseline on real FLAIR brain MRI (worst / median / best cases).
2. **3.2 Accuracy** — per-region Dice, HD95, and parameter count vs UNet3D, with the honest statistics.
3. **3.3 Uncertainty** — MC-Dropout calibration, and where the model is (over-)confident.

> **Read this first (honesty).** These are one run, no seeds, no confidence intervals. Wilcoxon signed-rank tests find **no region's Dice difference significant** (F01), and on the mean Dice OncoSeg and UNet3D essentially **tie** (OncoSeg ahead on 49/96 subjects, UNet3D on 47/96). The correct claim is *"matches UNet3D at ~5x fewer parameters,"* not *"beats it."* UNet3D was also OOM-killed at ~30 epochs, so the training budgets were unequal.

In [ ]:
# Runs IN-KERNEL so figures render inline. Helper: guarded PNG display with
# /content/oncoseg (Colab) path and a cwd fallback for a local clone.
import os
from IPython.display import Image, display

REPO = "/content/oncoseg" if os.path.isdir("/content/oncoseg") else os.getcwd()

def show_fig(rel_path, caption=None, width=1200):
    full = os.path.join(REPO, rel_path)
    if os.path.isfile(full):
        if caption:
            print(caption)
        display(Image(filename=full, width=width))
    else:
        print(f"[figure not found: {full}]")

print("### 3.1 - Segmentation on real validation brain MRI: OncoSeg vs UNet3D\n")
print("Rows = 3 representative cases (worst Dice=0.239 / median=0.852 / best=0.946).")
print("Columns = FLAIR input | expert ground truth | OncoSeg | UNet3D baseline (19.2M params).")
print("RGB overlay: Red=ET (enhancing) | Green=WT (whole tumor) | Blue=TC (tumor core).\n")
show_fig("figures/qualitative_comparison.png", None, width=1400)
print("\nReal MRI slices, not synthetic. Worst case shows the model can fail on")
print("heavily-necrotic tumors; median/best show it has learned robust 3D features.")

### 3.2 · How accurate is it?

OncoSeg reaches competitive Dice on BRATS (**n=96** validation subjects) with **~5.2x fewer parameters** than UNet3D (3.7M vs 19.2M), and a **lower** boundary error (HD95 15.35 mm vs 21.03 mm).

- **Per-region Dice (OncoSeg):** TC 0.790 · WT 0.853 · ET 0.748 · mean 0.797
- **Mean Dice:** 0.797 (OncoSeg) vs 0.794 (UNet3D) — a **statistical tie**
- **HD95 mean:** 15.35 mm (OncoSeg) vs 21.03 mm (UNet3D)
- **Parameters:** 3.7M vs 19.2M

**Honesty (F01).** Wilcoxon signed-rank (one-sided, OncoSeg > UNet3D) is non-significant in every region — **TC p=0.46, WT p=0.995, ET p=0.57, mean p=0.41**. On mean Dice OncoSeg leads on **49/96** subjects and UNet3D on **47/96** (a coin-flip). Single run, no seeds or confidence intervals, and UNet3D was OOM-killed at ~30 epochs (unequal budget). **Bottom line: OncoSeg matches UNet3D at ~5x smaller size** — a favorable efficiency trade, not a demonstrated accuracy win. A multi-seed, budget-matched comparison is needed before any "better" claim.

In [ ]:
# §3.2 accuracy: training curves + per-region Dice bar chart + a metrics table
# built straight from the committed eval JSONs. Reuses REPO/show_fig from §3.1.
import json
import pandas as pd

res = os.path.join(REPO, "experiments", "local_results")

print("### 3.2 - Accuracy\n")
print("Training curves (50 epochs) and per-region Dice comparison:\n")
show_fig("experiments/local_results/training_curves.png", None, width=1000)
show_fig("experiments/local_results/dice_comparison.png", None, width=1000)

try:
    with open(os.path.join(res, "oncoseg_eval.json")) as f:
        onc = json.load(f)
    with open(os.path.join(res, "unet3d_eval.json")) as f:
        unet = json.load(f)

    def row(label, ok, uk, fmt):
        ov, uv = onc[ok], unet[uk]
        return [label, f"{ov:{fmt}}", f"{uv:{fmt}}", f"{ov - uv:+{fmt[1:]}}"]

    tbl = [
        row("Dice TC",   "eval_dice_tc",   "eval_dice_tc",   ".4f"),
        row("Dice WT",   "eval_dice_wt",   "eval_dice_wt",   ".4f"),
        row("Dice ET",   "eval_dice_et",   "eval_dice_et",   ".4f"),
        row("Dice mean", "eval_dice_mean", "eval_dice_mean", ".4f"),
        row("HD95 mean (mm)", "eval_hd95_mean", "eval_hd95_mean", ".2f"),
    ]
    tbl.append(["Parameters", "3.7M", "19.2M", "~5.2x smaller"])
    tbl.append(["Val subjects", str(onc["num_val_subjects"]), str(unet["num_val_subjects"]), ""])
    df = pd.DataFrame(tbl, columns=["Metric", "OncoSeg", "UNet3D", "delta (Onco-UNet)"])
    print("\nQuantitative comparison (from committed eval JSONs):\n")
    print(df.to_string(index=False))

    print("\nStatistical significance (Wilcoxon signed-rank, one-sided OncoSeg>UNet3D):")
    print("  TC p=0.46 | WT p=0.995 | ET p=0.57 | mean p=0.41  -> none significant (F01)")
    print("  Mean Dice: OncoSeg wins 49/96, UNet3D 47/96 -> a tie, not a win.")
    print("  Single 50-epoch run; UNet3D OOM-killed ~30 epochs -> unequal budget.")
except FileNotFoundError as e:
    print(f"[eval JSON not found: {e}] - metrics table skipped.")

### 3.3 · Uncertainty & trustworthiness (MC-Dropout)

OncoSeg estimates per-voxel uncertainty with **Monte-Carlo Dropout** (5 stochastic forward passes; per-channel **binary** entropy, so it is bounded by ln 2 ≈ 0.69 nats — finding F16). Three views below:

1. **Uncertainty map** (median case BRATS_425): FLAIR, ground-truth mask, MC-Dropout entropy heatmap, and the prediction-error overlay. Uncertainty **concentrates on tumor boundaries** — where the model is genuinely unsure.
2. **Reliability diagram** (15-bin ECE): predicted confidence vs empirical accuracy.
3. **Uncertainty vs error**: higher entropy tracks higher per-voxel error, as expected.

**The calibration caveat (F02).** The **pooled ECE of 0.0101** looks excellent but is a background artifact — ~7.9M background voxels sit in the first bin (confidence ~0, accuracy ~0.002) and dominate the average. Restricted to tumor voxels, the **foreground ECE is 0.49**: the model is markedly **over-confident on the lesion voxels it gets wrong**. Clinically: use the uncertainty map to flag boundary regions for human review; do **not** read a high tumor-voxel confidence as a guarantee of correctness. As with §3.1–§3.2, these are static committed figures from a single run.

In [ ]:
# §3.3 uncertainty: three committed figures. Reuses REPO/show_fig from §3.1.
print("### 3.3 - Uncertainty quantification (MC-Dropout, 5 samples)\n")

figs = [
    ("figures/uncertainty_map.png",
     "Uncertainty map (BRATS_425): entropy concentrates at tumor boundaries."),
    ("figures/uncertainty_calibration.png",
     "Reliability diagram: pooled ECE=0.0101 (background-dominated) but "
     "FOREGROUND ECE=0.49 -> over-confident on tumor voxels (F02)."),
    ("figures/uncertainty_vs_error.png",
     "Uncertainty vs error: higher MC entropy tracks higher per-voxel error."),
]
for rel, cap in figs:
    print("\n" + cap)
    show_fig(rel, None, width=1000)

print("\nTakeaway: uncertainty is useful for flagging boundaries, but calibration is")
print("poor on tumor voxels (foreground ECE=0.49) - the voxels that matter most.")

## 4 · Full test suite

Runs as a subprocess (`!pytest`), so it always uses the freshly-installed packages. With the `dev,serve,dicom`
extras present, the tests that *skip* on a bare machine (needing monai/nibabel/pydicom/highdicom) now **run for
real**. Expectation: **all pass, 0 failed, 0 errors** (≈194 passed).

In [ ]:
!cd /content/oncoseg && pytest tests/ -q -rs --tb=short

## 5 · Lint (ruff) — same check CI runs

In [ ]:
!cd /content/oncoseg && ruff check src/ tests/ && echo 'ruff: clean'

## 6 · Smoke-test the fixed algorithm code paths (real tensors, no dataset)

Runs as a **subprocess** (fresh interpreter — immune to the stale-kernel issue). Exercises, on random
weights + synthetic volumes, that each fixed path *runs* (not that it's accurate — accuracy needs training):

- **F04** MC-Dropout on the trained *inline* `train_all.OncoSeg` (used to crash on `self.model.decoder`).
- **F16** uncertainty is per-channel **binary** entropy, bounded by `ln 2 ≈ 0.693`.
- **F06** RECIST longest diameter scans **all** slices (phantom: longest extent on a non-max-area slice).
- **F09** `DeepSupervisionLoss` interpolates multi-scale predictions instead of crashing.
- **F08** best-checkpoint selection is **NaN-safe**.

In [ ]:
smoke = r'''import sys, os
# train_all.py is a repo-root script (not an installed package module), so put
# the repo on sys.path before importing it.
sys.path.insert(0, os.getcwd())
import torch, numpy as np, math
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| numpy", np.__version__, "| torch", torch.__version__)
ok = True

# F04 + F16: MC-Dropout on the INLINE train_all architecture (the trained one)
from train_all import OncoSeg as InlineOncoSeg
from src.inference import Predictor
model = InlineOncoSeg(in_channels=4, num_classes=3, embed_dim=24, depths=(2,2,2,2),
                      num_heads=(3,6,12,24), deep_supervision=False).to(device).eval()
assert hasattr(model, "decoders") and not hasattr(model, "decoder")
pred = Predictor(model=model, device=torch.device(device), roi_size=(64,64,64), mc_samples=4)
unc = pred._estimate_uncertainty(torch.rand(1,4,64,64,64, device=device))  # F04: must not raise
f16 = unc.max() <= math.log(2) + 1e-3
print(f"F04 MC-dropout ran (shape {unc.shape}) -> OK")
print(f"F16 entropy<=ln2? max={float(unc.max()):.4f} (ln2={math.log(2):.4f}) -> {'OK' if f16 else 'FAIL'}"); ok &= f16

# F06: RECIST longest diameter across all slices
from src.response.recist import RECISTMeasurer
m = RECISTMeasurer()
mask = np.zeros((64,64,8), np.uint8); mask[10:40,10:40,0]=1; mask[30,5:55,1]=1
d = m.longest_axial_diameter(mask, pixdim=(1.0,1.0,1.0))
f06 = d > 45
print(f"F06 longest diameter={d:.1f}mm (expect ~49, pre-fix ~41) -> {'OK' if f06 else 'FAIL'}"); ok &= f06

# F09: deep-supervision loss interpolates multi-scale predictions
from src.training.losses import DeepSupervisionLoss, DiceCELoss
ds = DeepSupervisionLoss(DiceCELoss())
tgt = torch.zeros(1,3,32,32,32, device=device); tgt[:,0]=1
preds = [torch.randn(1,3,32,32,32, device=device), torch.randn(1,3,16,16,16, device=device), torch.randn(1,3,8,8,8, device=device)]
loss = ds(preds, tgt)  # F09: must not raise a shape error
f09 = bool(torch.isfinite(loss)) and loss.dim()==0
print(f"F09 deep-supervision loss={float(loss):.4f} finite scalar -> {'OK' if f09 else 'FAIL'}"); ok &= f09

# F08: NaN-safe best-checkpoint selection
guarded = lambda metric, best: (not math.isnan(metric)) and metric > best
row = np.array([0.71, 0.67, np.nan])  # empty-ET subject -> NaN region
f08 = math.isnan(float(np.mean(row))) and guarded(float(np.nanmean(row)), 0.0)
print(f"F08 plain-mean NaN, nanmean={float(np.nanmean(row)):.4f}, saves with guard -> {'OK' if f08 else 'FAIL'}"); ok &= f08

print("\nSMOKE_RESULT:", "ALL OK" if ok else "SOME FAILED")
sys.exit(0 if ok else 1)
'''
with open('/content/_smoke.py','w') as f: f.write(smoke)
!cd /content/oncoseg && python /content/_smoke.py

## 7 · Re-derive the two CRITICAL statistics from the committed arrays

No model needed — recomputes the honest numbers the docs now report (F01 Wilcoxon, F02 foreground ECE,
F07 dominant failure region) directly from the committed `.npy` / JSON. Also a subprocess.

In [ ]:
stats = r'''import numpy as np, json
from scipy.stats import wilcoxon
o = np.load("experiments/local_results/oncoseg_per_subject_dice.npy")  # cols [TC, WT, ET]
u = np.load("experiments/local_results/unet3d_per_subject_dice.npy")
print("per-subject arrays:", o.shape, "(val n =", o.shape[0], "-> split 388/96)")
for i,name in enumerate(["TC","WT","ET"]):
    a,b = o[:,i], u[:,i]; mk = ~(np.isnan(a)|np.isnan(b)); a,b = a[mk], b[mk]
    p = wilcoxon(a, b, alternative="greater").pvalue
    print(f"  {name}: delta={(a-b).mean():+.4f}  p(OncoSeg>UNet3D)={p:.4f}  OncoSeg wins {int((a>b).sum())}/{int(mk.sum())}")
om, um = np.nanmean(o,axis=1), np.nanmean(u,axis=1); mm = ~(np.isnan(om)|np.isnan(um))
print("  MEAN p =", round(float(wilcoxon(om[mm],um[mm],alternative="greater").pvalue),4), "-> F01: NO region significant; WT favors UNet3D")
means = np.nanmean(o, axis=1); bottom = np.argsort(means)[:5]
opr, bpr = np.nanmean(o,axis=0), np.nanmean(o[bottom],axis=0)
rel = {n:(opr[i]-bpr[i])/opr[i] for i,n in enumerate(["TC","WT","ET"])}
print("  F07 relative drop (bottom-5):", {k:round(v,3) for k,v in rel.items()}, "-> dominant =", max(rel, key=rel.get))
d = json.load(open("experiments/local_results/uncertainty_metrics.json"))
print("  F02 pooled ECE =", d["ece_median_case"], "| foreground ECE =", d.get("ece_median_case_foreground"), "-> over-confident on tumor")
'''
with open('/content/_stats.py','w') as f: f.write(stats)
!cd /content/oncoseg && python /content/_stats.py

## 8 · See it work — automated tumor tracking → RECIST 1.1 response (with figure)

This is the **"does the project actually do something?"** section, and it **renders an image inline** (it runs
in the kernel, not a subprocess, so the figure appears right below).

It takes a **baseline** tumor mask and three **follow-up** masks — shrinking, stable, growing — feeds each pair
through OncoSeg's **real** `RECISTMeasurer` + `ResponseClassifier`, and shows the clinical verdict
(CR / PR / SD / PD) two ways:

- **A results table** — baseline vs follow-up sum-of-longest-diameters, % change, and the verdict.
- **A before/after figure** — *top row:* baseline (blue) vs follow-up (orange) at the same slice, white =
  unchanged overlap, so you can literally see the tumor shrink or grow; *bottom row:* the follow-up
  segmentation with its automated verdict.

> **Honesty note.** The masks here are **synthetic sphere phantoms**, sized to straddle the RECIST thresholds,
> so the verdicts are correct *by construction*. This demonstrates the **measurement → classification code path**
> end-to-end — not model accuracy. The *identical* `classify()` call runs on real OncoSeg segmentations
> (`notebooks/recist_response_demo.ipynb` uses actual model predictions). The segmentation network itself is
> exercised in §6; a real end-to-end prediction needs a checkpoint you train first (§9), since none ships with the
> repo (finding F10).

In [ ]:
# Runs IN-KERNEL (not a subprocess) so the figure displays inline below.
import os, sys
sys.path.insert(0, "/content/oncoseg" if os.path.isdir("/content/oncoseg") else os.getcwd())
import numpy as np
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from src.response.recist import RECISTMeasurer
from src.response.classifier import ResponseClassifier, ResponseCategory

# Synthetic spherical "tumor" masks of known radius -- geometric phantoms, NOT
# model predictions. The point is to show the REAL RECIST 1.1 measurement +
# CR/PR/SD/PD classifier working end-to-end on a mask.
def sphere(dim=80, r=14):
    m = np.zeros((dim, dim, dim), np.uint8)
    c = dim // 2
    zz, yy, xx = np.ogrid[:dim, :dim, :dim]
    m[(zz - c) ** 2 + (yy - c) ** 2 + (xx - c) ** 2 <= r ** 2] = 1
    return m

clf, meas, pix = ResponseClassifier(), RECISTMeasurer(), (1.0, 1.0, 1.0)
baseline = sphere(r=14)                                  # ~28 mm target lesion
scenarios = {"Shrinking": sphere(r=9), "Stable": sphere(r=13), "Growing": sphere(r=19)}
arrow = {"Shrinking": "↓", "Stable": "≈", "Growing": "↑"}
glyph = {"Partial Response": "PR", "Stable Disease": "SD",
         "Progressive Disease": "PD", "Complete Response": "CR"}
colr  = {"Partial Response": "#1f9e89", "Stable Disease": "#b8860b",
         "Progressive Disease": "#d1362f", "Complete Response": "#1f9e89"}

base_les = meas.measure_lesions(baseline, pix)
base_sld = sum(l["longest_diameter_mm"] for l in base_les)
print("=== OncoSeg RECIST 1.1 response demo (synthetic phantoms, real measurement + classifier) ===")
print(f"baseline: {len(base_les)} target lesion(s), sum-longest-diameter = {base_sld:.1f} mm\n")
hdr = f"{'scenario':11s}{'baseline':>10s}{'follow-up':>11s}{'change':>9s}   verdict"
print(hdr); print("-" * len(hdr))
rows = []
for name, fu in scenarios.items():
    r = clf.classify(baseline, fu, pixdim=pix)
    rows.append((name, fu, r))
    v = r.category.value
    print(f"{name:11s}{r.baseline_sum_ld:8.1f}mm{r.followup_sum_ld:9.1f}mm{r.percent_change*100:+8.1f}%   {glyph[v]}  {v}")

def edge(slc):
    return ndi.binary_dilation(slc, iterations=1) & ~slc.astype(bool)

mid = baseline.shape[2] // 2
bs = baseline[:, :, mid]
fig, axes = plt.subplots(2, 4, figsize=(16, 9.5))
# column 0: baseline reference + legend
axes[0, 0].imshow(bs, cmap="gray"); axes[0, 0].contour(edge(bs), colors="#7CF6C8", linewidths=1.4)
axes[0, 0].set_title(f"BASELINE\nSLD = {base_sld:.0f} mm", fontsize=12, weight="bold"); axes[0, 0].axis("off")
axes[1, 0].axis("off")
axes[1, 0].legend(handles=[
    Patch(facecolor="#9ecae1", label="baseline tumor"),
    Patch(facecolor="#fdae6b", label="follow-up tumor"),
    Patch(facecolor="white", edgecolor="#999", label="overlap (unchanged)")],
    loc="center", fontsize=11, frameon=False, title="Top row = before/after overlay")
for j, (name, fu, r) in enumerate(rows, start=1):
    fs = fu[:, :, mid]
    ov = np.zeros((*bs.shape, 3))
    ov[bs > 0] = [0.62, 0.79, 0.88]                 # baseline = blue
    ov[fs > 0] = [0.99, 0.68, 0.42]                 # follow-up = orange
    ov[(bs > 0) & (fs > 0)] = [1, 1, 1]             # overlap = white
    axes[0, j].imshow(ov); axes[0, j].axis("off")
    axes[0, j].set_title(f"{name}  {arrow[name]}", fontsize=12, weight="bold")
    axes[1, j].imshow(fs, cmap="gray"); axes[1, j].contour(edge(fs), colors="#7CF6C8", linewidths=1.4)
    v = r.category.value
    axes[1, j].set_title(f"{glyph[v]}   {r.percent_change*100:+.0f}% SLD\n{v}",
                         fontsize=12.5, color=colr[v], weight="bold")
    axes[1, j].axis("off")
fig.suptitle("OncoSeg — automated 3D tumor tracking & RECIST 1.1 treatment-response",
             fontsize=15, weight="bold", y=1.0)
fig.text(0.5, 0.03,
         "Top: baseline (blue) vs follow-up (orange); white = unchanged overlap.   "
         "Bottom: follow-up segmentation + automated verdict.\n"
         "RECIST 1.1:  PR = shrink ≥30%   ·   PD = grow ≥20% (and ≥5 mm)   ·   SD = in between.   "
         "Synthetic phantoms — the identical code runs on real OncoSeg segmentations.",
         ha="center", fontsize=10, color="#555")
fig.subplots_adjust(hspace=0.28)
fig.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()

exp = {"Shrinking": ResponseCategory.PR, "Stable": ResponseCategory.SD, "Growing": ResponseCategory.PD}
print("\nDEMO_RESULT:", "ALL VERDICTS CORRECT" if all(r.category == exp[n] for n, _, r in rows) else "MISMATCH")

## 9 · (Optional, slow) Train end-to-end for a couple of epochs

Uncomment to exercise the **training loop** on the real MSD Brain Tumour dataset. Downloads **~7 GB** and
trains a few epochs on the GPU (tens of minutes). Verifies the seeded, NaN-guarded, correctly-labelled
training path runs end-to-end; it does **not** reproduce the paper's 50-epoch numbers.

In [ ]:
# # WARNING: downloads ~7GB and trains. Uncomment to run.
# !cd /content/oncoseg && python train_local.py --epochs 2 --val-interval 1 --seed 42

---
**How to read it:**
- **§3 Results Gallery** — real segmentation on brain MRI (OncoSeg vs UNet3D), the Dice/HD95/params table, and the MC-Dropout uncertainty figures. *This is the visual proof of what the project does.*
- **§4** — full test suite passes (incl. the monai/nibabel/pydicom/highdicom tests that skip on a bare machine); ≈194 passed, 0 failed.
- **§5** — `ruff: clean`.
- **§6** — `SMOKE_RESULT: ALL OK` (the fixed algorithm code paths run on real GPU tensors).
- **§7** — reproduces the honest headline statistics from the committed arrays.
- **§8** — prints a results table **and shows the before/after RECIST figure**, ending in `DEMO_RESULT: ALL VERDICTS CORRECT`.

If §3 doesn't show images, or §4/§6/§8 look off, paste the output back.